*Mount Google Drive. Change if using a different location*

In [1]:
# Change file path depending on the location of the data

from google.colab import drive
drive.mount('/content/drive')
BASE_PATH = "/content/drive/MyDrive/warehouse-bottleneck-analyzer"
MODELS_PATH = f"{BASE_PATH}/models"
RAW_DATA_PATH = f"{BASE_PATH}/data/raw/warehouse_line_throughput_synthetic.csv"
NOTEBOOKS_PATH = f"{BASE_PATH}/notebooks"

Mounted at /content/drive


*Data download and preprocessing*

In [2]:
import pandas as pd
import numpy as np

# Read the file, ensure "timestamp" is in date-time format
# then set it as the index.

df = pd.read_csv(RAW_DATA_PATH)
df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.sort_values("timestamp").reset_index(drop=True)
df = df.set_index("timestamp")

*After review of the data through a separate EDA program, **work_pressure** was the main feature driving operations's behavior. It is recreated here as a normalized function of **backlog_units** and **planned_work**. Managers care about today's backlog relative to today's planned workload. It is harder or impossible to staff when planned work is very high and capacity is near, or at, maximum. As backlog increases relative to planned work, there is more pressure to drive backlog to zero in the current hour*

In [3]:
# Engineered features
df["capacity_gap"] = (df["available_work"] - df["line_capacity"])
df["work_pressure"] = (df["backlog_units"] / df["planned_work"])

# Not used in this program but is available if needed.
#df["backlog_planned_ratio"] = (df["backlog_units"] / df["planned_work"])

# Print a few records and verify values of engineered features
print(df[[
    "available_work",
    "line_capacity",
    "capacity_gap",
    "work_pressure"
]].head())


                     available_work  line_capacity  capacity_gap  \
timestamp                                                          
2026-04-01 00:00:00           893.0            820          73.0   
2026-04-01 01:00:00           840.0            910         -70.0   
2026-04-01 02:00:00           696.0            640          56.0   
2026-04-01 03:00:00           804.0            672         132.0   
2026-04-01 04:00:00           846.0            539         307.0   

                     work_pressure  
timestamp                           
2026-04-01 00:00:00       0.116461  
2026-04-01 01:00:00       0.000000  
2026-04-01 02:00:00       0.030172  
2026-04-01 03:00:00       0.157088  
2026-04-01 04:00:00       0.448133  


*This helper function returns each model evaluation as a dictionary, with metric and hyperparameter names stored as keys. This provides a lightweight representation of one experiment, and allows the result to be inserted directly as a row in a Pandas DataFrame for side-by-side comparison across models.*

In [4]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    fbeta_score,
    roc_auc_score
)

def evaluate_model(model,
                   model_name,
                   parm_changed,
                   value,
                   y_true,
                   y_pred,
                   y_prob=None):
    """
    Returns a Dictionary containing evaluation metrics for one rf model.
    """

    results = {
        "Model Name": model_name,
        "Parm Changed": parm_changed,
        "Value": value,
        "n_estimators": model.n_estimators,
        "max_depth": model.max_depth,
        "min_samples_leaf": model.min_samples_leaf,
        "min_samples_split": model.min_samples_split,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
        "F1": f1_score(y_true, y_pred),
        "F2": fbeta_score(y_true, y_pred, beta=2)
    }

    if y_prob is not None:
        results["ROC AUC"] = roc_auc_score(y_true, y_prob)
    else:
        results["ROC AUC"] = "-"

    return (results)

*There are times where starting with an empty comparison table is needed. This progarm first used it prior to comparing classification metrics for a Random Forest model tuned with different hyperparameter values. It was expanded to compare a Logistic Regression model and a Random Forest model*

In [5]:
def reset_comparison_table():
    return pd.DataFrame(columns=[
        "Model Name",
        "Parm Changed",
        "Value",
        "n_estimators",
        "max_depth",
        "min_samples_leaf",
        "min_samples_split",
        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "F2",
        "ROC AUC"
    ])

In [6]:
comparison_table = reset_comparison_table()

*Starting with Random Forest as an arbitrary starting point, based only
on explanatory model efforts in an EDA program. It was initially believed there was a non-linear relationship between the target **next_hour_backlog** and the predictive features*

*It was decided the cost of missing backlog risks was higher than false positives. The model contained the feature set determined during EDA. An ablation study was performed to determine which removed features would not materially reduce the final model's ability to predict backlog risk. The study systematically removed one feature at a time and retrained the Random Forest model,the objective being to evaluate the individual contribution of each feature. Features whose removal did not materially reduce the Recall and F2 score metrics were identified as candidates for elimination. This drove model parsimony, resulting in a simpler and more interpretable model without sacrificing predictive accuracy.*

In [7]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

y = df["next_hour_backlog_risk"]
X = df.drop(columns=["next_hour_backlog"])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=85)

feature_sets = {
    "Full": [
        "work_pressure",
        "packers_assigned",
        "utilization_rate",
        "bottleneck_flag",
        "stations_down",
        "work_diverted_out"
    ],
    "No stations_down": [
        "work_pressure",
        "packers_assigned",
        "utilization_rate",
        "bottleneck_flag",
        "work_diverted_out"
    ],
    "No work_diverted_out": [
        "work_pressure",
        "packers_assigned",
        "utilization_rate",
        "bottleneck_flag"
    ],
    "No bottleneck_flag": [
        "work_pressure",
        "packers_assigned",
        "utilization_rate"
    ],
    "No packers_assigned": [
        "work_pressure",
        "utilization_rate"
    ],
    "No utilization_rate": [
        "work_pressure"
    ]
}

rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    min_samples_leaf=20,
    min_samples_split=20,
    random_state=7,
    class_weight="balanced"
    )

for model_name, features in feature_sets.items():

    rf.fit(
        X_train[features],
        y_train
    )

    y_pred = rf.predict(
        X_test[features]
    )

    experiment = evaluate_model(
      model=rf,
      model_name="Random Forest",
      parm_changed = "Feature Set",
      value = model_name,
      y_true=y_test,
      y_pred=y_pred
      )

    comparison_table.loc[len(comparison_table)] = experiment

In [8]:
display(comparison_table)

,Model Name,Parm Changed,Value,n_estimators,max_depth,min_samples_leaf,min_samples_split,Accuracy,Precision,Recall,F1,F2,ROC AUC
0,Random Forest,Feature Set,Full,100,5,20,20,0.733796,0.331081,0.753846,0.460094,0.600490,-
1,Random Forest,Feature Set,No stations_down,100,5,20,20,0.747685,0.349315,0.784615,0.483412,0.628079,-
2,Random Forest,Feature Set,No work_diverted_out,100,5,20,20,0.756944,0.355072,0.753846,0.482759,0.615578,-
3,Random Forest,Feature Set,No bottleneck_flag,100,5,20,20,0.733796,0.331081,0.753846,0.460094,0.600490,-
4,Random Forest,Feature Set,No packers_assigned,100,5,20,20,0.722222,0.322581,0.769231,0.454545,0.602410,-
5,Random Forest,Feature Set,No utilization_rate,100,5,20,20,0.726852,0.326797,0.769231,0.458716,0.605327,-


 *After the ablation study was performed on one train/test split, the selected feature set was validated using five-fold cross-validation to ensure that its performance generalized beyond a single train/test split. For comparison, the selected feature set was compared against the full feature set.*

*The selected feature set experienced nearly identical predictive performance, lower variability across folds, and benefitted from reduced complexity.*

In [9]:
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import make_scorer, fbeta_score

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=6
)

scoring = {
    "recall": "recall",
    "f2": make_scorer(
        fbeta_score,
        beta=2
    )
}

feature_sets = {
    "Full": [
        "work_pressure",
        "packers_assigned",
        "utilization_rate",
        "bottleneck_flag",
        "stations_down",
        "work_diverted_out"
    ],
    "Reduced_Model": [
        "work_pressure",
        "packers_assigned",
        "utilization_rate",
        "bottleneck_flag"
    ]
}

for model_name, features in feature_sets.items():

    scores = cross_validate(
        RandomForestClassifier(
            n_estimators=100,
            max_depth=5,
            min_samples_leaf=20,
            min_samples_split=20,
            random_state=7,
            class_weight="balanced"
            ),
        X[features],
        y,
        cv=cv,
        scoring=scoring
    )

    print(
        model_name,
        scores["test_recall"].mean(),
        scores["test_recall"].std(),
        scores["test_f2"].mean(),
        scores["test_f2"].std()
    )


Full 0.8265827471306924 0.024592072833029194 0.6720605446414959 0.010188615494683497
Reduced_Model 0.8103295075897815 0.016736262783763882 0.6702027068216495 0.007662174114467063


*This code section expanded the cross-validation by showing the differences between the full and reduced models for each fold. The goal is to verify if the reduction in performance was consistent and modest across the folds, rather than being caused by a few unusually poor folds.*

In [10]:
full_features = [
    "work_pressure",
    "packers_assigned",
    "utilization_rate",
    "bottleneck_flag",
    "stations_down",
    "work_diverted_out"
]

reduced_features = [
    "work_pressure",
    "packers_assigned",
    "utilization_rate",
    "bottleneck_flag"
]

full_scores = cross_validate(
    RandomForestClassifier(
        n_estimators=100,
        max_depth=5,
        min_samples_leaf=20,
        min_samples_split=20,
        random_state=7,
        class_weight="balanced"
    ),
    X[full_features],
    y,
    cv=cv,
    scoring=scoring
)

reduced_scores = cross_validate(
    RandomForestClassifier(
        n_estimators=100,
        max_depth=5,
        min_samples_leaf=20,
        min_samples_split=20,
        random_state=7,
        class_weight="balanced"
    ),
    X[reduced_features],
    y,
    cv=cv,
    scoring=scoring
)

fold_results = pd.DataFrame({
    "Fold": range(1, len(full_scores["test_recall"]) + 1),
    "Full_Recall": full_scores["test_recall"],
    "Reduced_Recall": reduced_scores["test_recall"],
    "Difference": full_scores["test_recall"] - reduced_scores["test_recall"]
})

print(fold_results.round(4))

   Fold  Full_Recall  Reduced_Recall  Difference
0     1       0.8514          0.8243      0.0270
1     2       0.7973          0.7973      0.0000
2     3       0.8514          0.8243      0.0270
3     4       0.7973          0.7838      0.0135
4     5       0.8356          0.8219      0.0137


*We'll start with the reduced set of features, and manually tune a standard set of hyperparameters as an exploratory exercise. While known not to be globally optimal, the exercise isolates the influence of each individual hyperparameter. The results of each pass will be captured, and we'll compare their Recall and F2-score scores. We'll keep each set of hyperparameters constant while we'll change one at a time.*

In [11]:
X = df[reduced_features]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=85)

rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

comparison_table = reset_comparison_table()

*This code shows the result of training the baseline model. Each item in the variable **experiment** holds the classification report metric. It is provided to validate the returned dictionary contains the values needed.*

In [12]:
# Create a new result row and add the evaluation
experiment = evaluate_model(
    model=rf,
    model_name="Random Forest",
    parm_changed = "Default",
    value = "Baseline",
    y_true=y_test,
    y_pred=y_pred
)

for key, value in experiment.items():
    if isinstance(value, float):
        print(f"{key:12}: {value:.3f}")
    else:
        print(f"{key:12}: {value}")

comparison_table.loc[len(comparison_table)] = experiment

Model Name  : Random Forest
Parm Changed: Default
Value       : Baseline
n_estimators: 100
max_depth   : 5
min_samples_leaf: 20
min_samples_split: 20
Accuracy    : 0.757
Precision   : 0.355
Recall      : 0.754
F1          : 0.483
F2          : 0.616
ROC AUC     : -


*The variable **experiment** will now house each prediction result for each changed hyperparameter value. **Experiment** will be loaded into a dataframe for comparison.*

*We start with tuning **n_estimators**.*

In [13]:
for change_n_estimators in [150, 200, 250, 300, 350, 400, 450, 500]:
    rf = RandomForestClassifier(
        n_estimators=change_n_estimators,
        max_depth=5,
        min_samples_leaf=20,
        min_samples_split=20,
        random_state=789,
        class_weight="balanced"
        )

    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_test)

    experiment = evaluate_model(
        model=rf,
        model_name="Random Forest",
        parm_changed = "n_estimators",
        value = change_n_estimators,
        y_true=y_test,
        y_pred=y_pred
      )

    comparison_table.loc[len(comparison_table)] = experiment

*In this case, **n_estimator** values of 150 and 250 resulted in the highest values for Recall and F2-score, even though Recall remained effectively unchanged, indicating that increasing the number of trees did not improve the model's ability to identify backlog-risk periods.*

*As the value of 150 provided the same results with lower computational costs, that value will be used in subsequent tuning.*

In [14]:
display(comparison_table)

,Model Name,Parm Changed,Value,n_estimators,max_depth,min_samples_leaf,min_samples_split,Accuracy,Precision,Recall,F1,F2,ROC AUC
0,Random Forest,Default,Baseline,100,5,20,20,0.756944,0.355072,0.753846,0.482759,0.615578,-
1,Random Forest,n_estimators,150,150,5,20,20,0.754630,0.356643,0.784615,0.490385,0.632754,-
2,Random Forest,n_estimators,200,200,5,20,20,0.752315,0.352113,0.769231,0.483092,0.621891,-
3,Random Forest,n_estimators,250,250,5,20,20,0.754630,0.356643,0.784615,0.490385,0.632754,-
4,Random Forest,n_estimators,300,300,5,20,20,0.754630,0.354610,0.769231,0.485437,0.623441,-
5,Random Forest,n_estimators,350,350,5,20,20,0.756944,0.355072,0.753846,0.482759,0.615578,-
6,Random Forest,n_estimators,400,400,5,20,20,0.759259,0.357664,0.753846,0.485149,0.617128,-
7,Random Forest,n_estimators,450,450,5,20,20,0.759259,0.357664,0.753846,0.485149,0.617128,-
8,Random Forest,n_estimators,500,500,5,20,20,0.759259,0.357664,0.753846,0.485149,0.617128,-


*Tuning **max_depth** following the same pattern as was done for **n_estimators**.*

In [15]:
for change_max_depth in [8, 11, 14, 17, 20]:
    rf = RandomForestClassifier(
        n_estimators=100,
        max_depth=change_max_depth,
        min_samples_leaf=20,
        min_samples_split=20,
        random_state=2121,
        class_weight="balanced",
        n_jobs=-1
        )

    experiment = evaluate_model(
        model=rf,
        model_name="Random Forest",
        parm_changed = "Max Depth",
        value = change_max_depth,
        y_true=y_test,
        y_pred=y_pred
      )

    comparison_table.loc[len(comparison_table)] = experiment

*In this case, **max_depth** was identical to the baseline for all classification report values execpt for F2-score being a little higher. Because a smaller max_depth value requires less training time and less memory use, the trees are simpler, with lower overfitting risk and easier interpretations, a value of 8 was chosen.*

In [16]:
display(comparison_table)

,Model Name,Parm Changed,Value,n_estimators,max_depth,min_samples_leaf,min_samples_split,Accuracy,Precision,Recall,F1,F2,ROC AUC
0,Random Forest,Default,Baseline,100,5,20,20,0.756944,0.355072,0.753846,0.482759,0.615578,-
1,Random Forest,n_estimators,150,150,5,20,20,0.754630,0.356643,0.784615,0.490385,0.632754,-
2,Random Forest,n_estimators,200,200,5,20,20,0.752315,0.352113,0.769231,0.483092,0.621891,-
3,Random Forest,n_estimators,250,250,5,20,20,0.754630,0.356643,0.784615,0.490385,0.632754,-
4,Random Forest,n_estimators,300,300,5,20,20,0.754630,0.354610,0.769231,0.485437,0.623441,-
5,Random Forest,n_estimators,350,350,5,20,20,0.756944,0.355072,0.753846,0.482759,0.615578,-
6,Random Forest,n_estimators,400,400,5,20,20,0.759259,0.357664,0.753846,0.485149,0.617128,-
7,Random Forest,n_estimators,450,450,5,20,20,0.759259,0.357664,0.753846,0.485149,0.617128,-
8,Random Forest,n_estimators,500,500,5,20,20,0.759259,0.357664,0.753846,0.485149,0.617128,-
9,Random Forest,Max Depth,8,100,8,20,20,0.759259,0.357664,0.753846,0.485149,0.617128,-


*Tune **min_samples_leaf**.*

In [17]:
for change_min_samples_leaf in [1, 5, 9, 13, 17, 20]:
    rf = RandomForestClassifier(
        n_estimators=100,
        max_depth=5,
        min_samples_leaf=change_min_samples_leaf,
        min_samples_split=20,
        random_state=2121,
        class_weight="balanced",
        n_jobs=-1
        )

    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_test)

    experiment = evaluate_model(
        model=rf,
        model_name="Random Forest",
        parm_changed = "Minimum Samples Leaf",
        value = change_min_samples_leaf,
        y_true=y_test,
        y_pred=y_pred
      )

    comparison_table.loc[len(comparison_table)] = experiment

*Increasing **min_samples_leaf** improved Recall and F2 score. This suggests larger leaves promoted better generalization to unseen data. Although the leaf sizes may overlook highly localized patterns, they reduce the risk of overfitting.*
*Because the highest Recall and F2 scores were obtained with **min_samples_leaf** = 17, this value was retained for subsequent model tuning.*

In [18]:
display(comparison_table)

,Model Name,Parm Changed,Value,n_estimators,max_depth,min_samples_leaf,min_samples_split,Accuracy,Precision,Recall,F1,F2,ROC AUC
0,Random Forest,Default,Baseline,100,5,20,20,0.756944,0.355072,0.753846,0.482759,0.615578,-
1,Random Forest,n_estimators,150,150,5,20,20,0.754630,0.356643,0.784615,0.490385,0.632754,-
2,Random Forest,n_estimators,200,200,5,20,20,0.752315,0.352113,0.769231,0.483092,0.621891,-
3,Random Forest,n_estimators,250,250,5,20,20,0.754630,0.356643,0.784615,0.490385,0.632754,-
4,Random Forest,n_estimators,300,300,5,20,20,0.754630,0.354610,0.769231,0.485437,0.623441,-
5,Random Forest,n_estimators,350,350,5,20,20,0.756944,0.355072,0.753846,0.482759,0.615578,-
6,Random Forest,n_estimators,400,400,5,20,20,0.759259,0.357664,0.753846,0.485149,0.617128,-
7,Random Forest,n_estimators,450,450,5,20,20,0.759259,0.357664,0.753846,0.485149,0.617128,-
8,Random Forest,n_estimators,500,500,5,20,20,0.759259,0.357664,0.753846,0.485149,0.617128,-
9,Random Forest,Max Depth,8,100,8,20,20,0.759259,0.357664,0.753846,0.485149,0.617128,-


*Tune **min_samples_split**.*

In [19]:
for change_min_samples_split in [2, 5, 8, 11, 14, 17, 20]:
    rf = RandomForestClassifier(
        n_estimators=100,
        max_depth=5,
        min_samples_leaf=20,
        min_samples_split=change_min_samples_split,
        random_state=32,
        class_weight="balanced",
        n_jobs=-1
        )

    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_test)

    experiment = evaluate_model(
        model=rf,
        model_name="Random Forest",
        parm_changed = "Minimum Samples Split",
        value = change_min_samples_split,
        y_true=y_test,
        y_pred=y_pred
      )

    comparison_table.loc[len(comparison_table)] = experiment

*Seeing that the metrics were identical, model performance was largely insensitive to **min_samples_split** when tuned independently. That suggests other tree-growth constraints had a greater influence under the baseline settings. Just for this study, a large **min_samples_split** was chosen because it can reduce computational cost.*

In [20]:
display(comparison_table)

,Model Name,Parm Changed,Value,n_estimators,max_depth,min_samples_leaf,min_samples_split,Accuracy,Precision,Recall,F1,F2,ROC AUC
0,Random Forest,Default,Baseline,100,5,20,20,0.756944,0.355072,0.753846,0.482759,0.615578,-
1,Random Forest,n_estimators,150,150,5,20,20,0.754630,0.356643,0.784615,0.490385,0.632754,-
2,Random Forest,n_estimators,200,200,5,20,20,0.752315,0.352113,0.769231,0.483092,0.621891,-
3,Random Forest,n_estimators,250,250,5,20,20,0.754630,0.356643,0.784615,0.490385,0.632754,-
4,Random Forest,n_estimators,300,300,5,20,20,0.754630,0.354610,0.769231,0.485437,0.623441,-
5,Random Forest,n_estimators,350,350,5,20,20,0.756944,0.355072,0.753846,0.482759,0.615578,-
6,Random Forest,n_estimators,400,400,5,20,20,0.759259,0.357664,0.753846,0.485149,0.617128,-
7,Random Forest,n_estimators,450,450,5,20,20,0.759259,0.357664,0.753846,0.485149,0.617128,-
8,Random Forest,n_estimators,500,500,5,20,20,0.759259,0.357664,0.753846,0.485149,0.617128,-
9,Random Forest,Max Depth,8,100,8,20,20,0.759259,0.357664,0.753846,0.485149,0.617128,-


*Compare tuned parameters against baseline*

In [21]:
# Uncomment to show just baseline and tuned
comparison_table = reset_comparison_table()

rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    min_samples_leaf=20,
    min_samples_split=20,
    random_state=789,
    class_weight="balanced"
)

rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

experiment = evaluate_model(
    model=rf,
    model_name="Random Forest",
    parm_changed = "Default",
    value = "Baseline",
    y_true=y_test,
    y_pred=y_pred
    )

comparison_table.loc[len(comparison_table)] = experiment

rf = RandomForestClassifier(
    n_estimators=150,
    max_depth=8,
    min_samples_leaf=17,
    min_samples_split=20,
    random_state=44,
    class_weight="balanced",
    n_jobs=-1
    )

rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

experiment = evaluate_model(
    model=rf,
    model_name="Random Forest",
    parm_changed = "Tier 1 parms",
    value = "Tuned",
    y_true=y_test,
    y_pred=y_pred
    )

comparison_table.loc[len(comparison_table)] = experiment

display(comparison_table)

,Model Name,Parm Changed,Value,n_estimators,max_depth,min_samples_leaf,min_samples_split,Accuracy,Precision,Recall,F1,F2,ROC AUC
0,Random Forest,Default,Baseline,100,5,20,20,0.754630,0.356643,0.784615,0.490385,0.632754,-
1,Random Forest,Tier 1 parms,Tuned,150,8,17,20,0.777778,0.381679,0.769231,0.510204,0.639386,-


*With cross-validation scoring, we'll answer the questions about how well the model performs on average and its consistency. The results indicate the model behaves quite consistently on both Recall and F2 scores. There is the caveat that the cross-validation effort was only 5-fold, and therefore isn't direct proof of stability.*

In [22]:
scores = cross_validate(
    RandomForestClassifier(
        n_estimators=300,
        max_depth=8,
        min_samples_leaf=17,
        min_samples_split=20,
        random_state=7,
        class_weight="balanced"
    ),
    X,
    y,
    cv=cv,
    scoring=scoring
)
print(
    "Test Recall Mean: ",scores["test_recall"].mean(),
    "\nTest Recall Standard Deviation: ",scores["test_recall"].std(),
    "\nTest F2 Mean: ",scores["test_f2"].mean(),
    "\nTest F2 Standard Deviation: ",scores["test_f2"].std()
    )

Test Recall Mean:  0.7833024805627545 
Test Recall Standard Deviation:  0.03362496586682094 
Test F2 Mean:  0.6614410668934408 
Test F2 Standard Deviation:  0.014382753841944948


*Now that the manual determination of best hyper-parameter values, the next effort will introduce GridSearch. It evaluated every combination of values specified in the parameter grid (the grid was kept small to control the number of fits and reduce the computational cost). For each combination, five-fold cross-validation was performed and the mean F2 score across the validation folds was calculated. The hyperparameter combination producing the highest mean cross-validated F2 score was selected as the optimal model configuration.*

In [23]:
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import StratifiedKFold

cv_strategy = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=3
)

# F2 scorer
f2_scorer = make_scorer(
    fbeta_score,
    beta=2,
    pos_label=1,
    zero_division=0)

# Base model
rf = RandomForestClassifier(
    random_state=1111,
    class_weight="balanced",
    n_jobs=-1)

# Parameter grid
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [8, 14, 20],
    'min_samples_leaf': [13, 17, 20],
    'min_samples_split': [2, 5, 8]
}

# Grid search
grid = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    scoring=f2_scorer,
    cv=cv_strategy,
    n_jobs=-1,
    verbose=2
)

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

print("\nFeatures:")
print(X_train.columns.tolist())

print("\nTarget distribution:")
print(y_train.value_counts())
print(y_train.value_counts(normalize=True))

# Fit to the training data only
grid.fit(X_train, y_train)

print("Scorer:", grid.scorer_)
print("Best Score:", grid.best_score_)
print("Best Parameters:", grid.best_params_)
print("Best CV F2:", grid.best_score_)

X_train shape: (1727, 4)
y_train shape: (1727,)

Features:
['work_pressure', 'packers_assigned', 'utilization_rate', 'bottleneck_flag']

Target distribution:
next_hour_backlog_risk
0    1423
1     304
Name: count, dtype: int64
next_hour_backlog_risk
0    0.823972
1    0.176028
Name: proportion, dtype: float64
Fitting 5 folds for each of 81 candidates, totalling 405 fits
Scorer: make_scorer(fbeta_score, response_method='predict', beta=2, pos_label=1, zero_division=0)
Best Score: 0.6730882388633292
Best Parameters: {'max_depth': 8, 'min_samples_leaf': 20, 'min_samples_split': 2, 'n_estimators': 300}
Best CV F2: 0.6730882388633292


*From the previous code block, the best estimator was determined by the hyperparameter combination that achieved the highest mean cross-validated F2 score. Using this model the metrics Recall, Precision, and F1, were computed to provide a more comprehensive evaluation of its predictive performance.*

In [24]:
best_rf = grid.best_estimator_

scoring = {
    "precision": make_scorer(precision_score, zero_division=0),
    "recall": make_scorer(recall_score, zero_division=0),
    "f1": make_scorer(f1_score, zero_division=0),
    "f2": make_scorer(fbeta_score, beta=2, zero_division=0)
}

best_cv_results = cross_validate(
    best_rf,
    X_train,
    y_train,
    cv=cv_strategy,
    scoring=scoring,
    n_jobs=-1
)

for metric in ["precision", "recall", "f1", "f2"]:
    scores = best_cv_results[f"test_{metric}"]
    print(
        f"{metric.capitalize()} Mean: {scores.mean():.4f} "
        f"| Std: {scores.std():.4f}"
    )

Precision Mean: 0.4093 | Std: 0.0242
Recall Mean: 0.8026 | Std: 0.0259
F1 Mean: 0.5420 | Std: 0.0269
F2 Mean: 0.6731 | Std: 0.0274


*In this code block, the best model was trained, predications calculated, and the metrics were displayed using a classification report.*

In [25]:
from sklearn.metrics import classification_report

best_rf = grid.best_estimator_
best_rf.fit(X_train, y_train)

y_test_pred = best_rf.predict(X_test)

print(classification_report(
    y_test,
    y_test_pred,
    target_names=["No Backlog Risk", "Backlog Risk"],
    digits=4
))

test_f2 = fbeta_score(
    y_test,
    y_test_pred,
    beta=2,
    zero_division=0
)

print(f"Final Test F2: {test_f2:.4f}")
print(f"Final ROC AUC: {roc_auc_score(y_test, y_test_pred):.4f}")

                 precision    recall  f1-score   support

No Backlog Risk     0.9467    0.7738    0.8516       367
   Backlog Risk     0.3712    0.7538    0.4975        65

       accuracy                         0.7708       432
      macro avg     0.6589    0.7638    0.6745       432
   weighted avg     0.8601    0.7708    0.7983       432

Final Test F2: 0.6250
Final ROC AUC: 0.7638


*After selecting the features using Random Forest, create baseline results with Logistic Regression for comparison. The goal is to verify the feature relationships are non-linear enough to choose a model different that Random Forest.*

*I chose ColumnTransformer in case I expand the number of features later. Instead of manually calling StandardScalar and other functions, I kept them together and let ColumnTransformer handle the calls. An additional benefit is it makes it less error-prone. It will be used in a Logistic Regression model pipeline.*

In [26]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

continuous_features = [
    "work_pressure",
    "packers_assigned",
    "utilization_rate"
]

binary_features = [
    "bottleneck_flag"
]

preprocessor = ColumnTransformer(
    transformers=[
        ("continuous", StandardScaler(), continuous_features),
        ("binary", "passthrough", binary_features)
    ],
    # This will remove the words "continuous" and "binary" from the feature names
    # and if the function get_feature_names_out is called it will return the clean feature names
    verbose_feature_names_out=False
)

*This code block creates and fits a Logistic Regression model. Its Recall and F2 score will be compared later against results from the Random Forest model.*

*An L2 penalty was specified to help reduce the possibility of overfitting due to large coefficients, even though during explanatory analysis (performed in another program) the final coefficient estimates were moderate and the model used only three predictors.Liblinear was specified for the solver as the feature set is quite small and works for binary classification models.*

In [27]:
lr_model = Pipeline([
    ("preprocessor", preprocessor),

    ("classifier",
        LogisticRegression(
            class_weight="balanced",
            penalty="l2",
            solver="liblinear",
            max_iter=1000,
            random_state=6
        )
    )
])

lr_model.fit(X_train, y_train)

y_pred = lr_model.predict(X_test)

print(classification_report(
    y_test,
    y_pred,
    target_names=["No Backlog Risk", "Backlog Risk"],
    digits=4
))

test_f2 = fbeta_score(
    y_test,
    y_pred,
    beta=2
)

print(f"Final Test F2: {test_f2:.4f}")
print(f"Final ROC AUC: {roc_auc_score(y_test, y_pred):.4f}")

                 precision    recall  f1-score   support

No Backlog Risk     0.9524    0.7629    0.8472       367
   Backlog Risk     0.3696    0.7846    0.5025        65

       accuracy                         0.7662       432
      macro avg     0.6610    0.7738    0.6748       432
   weighted avg     0.8647    0.7662    0.7953       432

Final Test F2: 0.6407
Final ROC AUC: 0.7738


*As was done for the Random Forest model, cross-validation was performed for the Logistic Regression model. Not only does it provide how well the LR model generalizes and how stable it is, but also provides a fair and consistent evaluation framework for comparing against the Random Forest model.*

In [28]:
scoring = {
    "precision": make_scorer(precision_score),
    "recall": make_scorer(recall_score),
    "f1": make_scorer(f1_score),
    "f2": make_scorer(fbeta_score, beta=2)
}

cv_results = cross_validate(
    lr_model,
    X_train,
    y_train,
    cv=cv_strategy,
    scoring=scoring,
    n_jobs=-1
)

print()

for metric in scoring.keys():
    scores = cv_results[f"test_{metric}"]

    print(
        f"{metric.capitalize():10}"
        f" Mean = {scores.mean():.4f}"
        f"   Std = {scores.std():.4f}"
    )


Precision  Mean = 0.4248   Std = 0.0182
Recall     Mean = 0.8026   Std = 0.0298
F1         Mean = 0.5554   Std = 0.0203
F2         Mean = 0.6811   Std = 0.0234


*As we used a pipeline, "classifier" was one of the named steps. Earlier we put the feature names with the order "work_pressure", "packers_assigned", "utilization_rate" and "bottleneck_flag". As we sort by largest coefficient, "bottleneck_flag" appears second, even it represents beta_sub_4 in the logit function*

*What's nice about using the function "get_feature_names_out" is that it's scalable if the model adds additional features, especially if they are different than continuous or binary types*

In [29]:
feature_names = preprocessor.get_feature_names_out()

coefficients = lr_model.named_steps[
    "classifier"
].coef_[0]

coef_df = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": coefficients,
    "Abs Coefficient": np.abs(coefficients)
})

coef_df = coef_df.sort_values(
    "Abs Coefficient",
    ascending=False
)

print(coef_df)

            Feature  Coefficient  Abs Coefficient
0     work_pressure     1.782523         1.782523
3   bottleneck_flag     1.015658         1.015658
1  packers_assigned     0.532693         0.532693
2  utilization_rate    -0.109275         0.109275


*Note that the odds ratio for utilization rate is less than 1, which means a one standard deviation (because StandardScalar was used) increase in utilization_rate is associated with a decrease in the odds of backlog risk (with 1 - 0.8965 = 0.1035, it's a little more than 10% decrease), holding all other predictors constant.*

In [30]:
# Extract the fitted logistic regression model
lr_classifier = lr_model.named_steps["classifier"]

# Extract coefficients for the positive class Backlog Risk = 1
coefficients = lr_classifier.coef_[0]

coefficient_df = pd.DataFrame({
    "Feature": feature_names, # already retrieved from get_feature_names_out()
    "Coefficient": coefficients,
    "Absolute_Coefficient": np.abs(coefficients),
    "Odds_Ratio": np.exp(coefficients)
})

coefficient_df = coefficient_df.sort_values(
    by="Absolute_Coefficient",
    ascending=False
).reset_index(drop=True)

print(coefficient_df.round(4))

            Feature  Coefficient  Absolute_Coefficient  Odds_Ratio
0     work_pressure       1.7825                1.7825      5.9448
1   bottleneck_flag       1.0157                1.0157      2.7612
2  packers_assigned       0.5327                0.5327      1.7035
3  utilization_rate      -0.1093                0.1093      0.8965


*McNemar's test is run on both models to know whether one model is more accurate than the other on the same observations, or whether the observed differences could simply be due to chance. It focuses on the two disagreements between them: when the RF model is correct and the LR model is incorrect, and vice versa. We want to see if those two counts differ more than we would reasonably expect from sampling variation.*

*In this case, because the disagreement numbers, 9 and 11, are nearly balanced, the test found no evidence that either model was more likely to classify observations correctly.*

*I passed exact=true to the mcnemar's test because I've read references that recommend it whenever the number of discordant pairs is relatively small (20) in my case*


In [31]:
from statsmodels.stats.contingency_tables import mcnemar

# Random Forest
y_pred_rf = best_rf.predict(X_test)

# Logistic Regression
y_pred_lr = lr_model.predict(X_test)

rf_correct = (y_pred_rf == y_test)
lr_correct = (y_pred_lr == y_test)

table = pd.DataFrame(
    [
        [
            ((~rf_correct) & (~lr_correct)).sum(),
            ((~rf_correct) & ( lr_correct)).sum()
        ],
        [
            (( rf_correct) & (~lr_correct)).sum(),
            (( rf_correct) & ( lr_correct)).sum()
        ]
    ],
    index=["RF Incorrect", "RF Correct"],
    columns=["LR Incorrect", "LR Correct"]
)

print(table)

numpy_table = table.to_numpy()

result = mcnemar(numpy_table, exact=True)

print(f"\nStatistic: {result.statistic}")
print(f"P-value: {result.pvalue:.4f}")

              LR Incorrect  LR Correct
RF Incorrect            90           9
RF Correct              11         322

Statistic: 9.0
P-value: 0.8238


*From the results containing the coefficients and odds ratios displayed earlier, I wanted to determine whether each additional predictor contributed enough information to justify its inclusion in the final Logistic Regression model. A sequence of nested Logistic Regression models was fitted using progressively larger feature sets, starting with the feature **work_pressure**.*

In [32]:
import statsmodels.api as sm

models = {}

feature_sets = {
    "LR1": ["work_pressure"],
    "LR2": ["work_pressure",
            "bottleneck_flag"],
    "LR3": [
        "work_pressure",
        "bottleneck_flag",
        "packers_assigned"],
    "LR4": [
        "work_pressure",
        "bottleneck_flag",
        "packers_assigned",
        "utilization_rate"
    ]
}

for name, features in feature_sets.items():
    X = sm.add_constant(df[features])
    model = sm.Logit(df["next_hour_backlog_risk"], X).fit(disp=False)
    models[name] = model

*I ran chi-squared tests on each successive nested Logit model. As a result, adding **bottleneck_flag** fit the data better than just **work_pressure** alone. Adding **packers_assigned** greatly fit the data better than the 2 featured-model. Finally, adding the feature **utilization_rate** only resulted in a small improvement that was not statistically significant.*

In [33]:
from scipy.stats import chi2

comparisons = []

model_names = list(models.keys())

for i in range(1, len(model_names)):
    reduced = models[model_names[i-1]]
    full = models[model_names[i]]

    lr_stat = 2 * (full.llf - reduced.llf)
    df_diff = full.df_model - reduced.df_model
    p_value = chi2.sf(lr_stat, df_diff)

    comparisons.append({
        "Comparison": f"{model_names[i-1]} vs {model_names[i]}",
        "LR Chi-square": lr_stat,
        "df": df_diff,
        "p-value": p_value
    })

chi_square_df = pd.DataFrame(comparisons)

chi_square_df["p-value"] = chi_square_df["p-value"].apply(
    lambda x: "< 0.001" if x < 0.001 else f"{x:.4f}"
)

print(chi_square_df)

   Comparison  LR Chi-square   df  p-value
0  LR1 vs LR2       9.008115  1.0   0.0027
1  LR2 vs LR3      56.353147  1.0  < 0.001
2  LR3 vs LR4       1.662510  1.0   0.1973


*Adding to the argument not to include feature **utilization_rate** to the model, while the log-likelihood improves, the model's AIC and BIC values say the improvement is too small to justify the added complexity.*

In [34]:
lr_model_summary = []

for name, model in models.items():

    lr_model_summary.append({
        "Model": name,
        "Features": len(model.params) - 1,
        "Log-Likelihood": model.llf,
        "AIC": model.aic,
        "BIC": model.bic
    })

print(pd.DataFrame(lr_model_summary))

  Model  Features  Log-Likelihood          AIC          BIC
0   LR1         1     -727.213745  1458.427489  1469.782290
1   LR2         2     -722.709687  1451.419374  1468.451575
2   LR3         3     -694.533114  1397.066227  1419.775829
3   LR4         4     -693.701859  1397.403718  1425.790720


*A new model is fit with just **work_pressure**, **packers_assigned**, and **bottleneck_flag**, and its classification metrics are provided.*

In [35]:
continuous_features2 = [
    "work_pressure",
    "packers_assigned"
]

preprocessor2 = ColumnTransformer(
    transformers=[
        ("continuous", StandardScaler(), continuous_features2),
        ("binary", "passthrough", binary_features)
    ]
)

lr_model2 = Pipeline([
    ("preprocessor", preprocessor2),

    ("classifier",
        LogisticRegression(
            class_weight="balanced",
            penalty="l2",
            solver="liblinear",
            max_iter=1000,
            random_state=6
        )
    )
])

lr_model2.fit(X_train, y_train)

y_pred2 = lr_model2.predict(X_test)

print(classification_report(
    y_test,
    y_pred2,
    target_names=["No Backlog Risk", "Backlog Risk"],
    digits=4
))

test2_f2 = fbeta_score(
    y_test,
    y_pred2,
    beta=2
)

print(f"Final Test F2: {test2_f2:.4f}")
print(f"Final ROC AUC: {roc_auc_score(y_test, y_pred2):.4f}")

                 precision    recall  f1-score   support

No Backlog Risk     0.9524    0.7629    0.8472       367
   Backlog Risk     0.3696    0.7846    0.5025        65

       accuracy                         0.7662       432
      macro avg     0.6610    0.7738    0.6748       432
   weighted avg     0.8647    0.7662    0.7953       432

Final Test F2: 0.6407
Final ROC AUC: 0.7738


*I wanted to show how much the predicted probabilities changed if the feature **utilization_rate** was removed. This is shown with Maximum absolute, mean, median, and 95 percentile differences. At best the maximum change was about 5%, the mean about 1%, the 95 percentile about 3.5%, and around the median the probabilites were nearly identical*

In [36]:
prob_lr3 = lr_model2.predict_proba(X_test)[:, 1]
prob_lr4 = lr_model.predict_proba(X_test)[:, 1]

print(f"Maximum absolute difference: {np.max(np.abs(prob_lr3 - prob_lr4)):.4f}")

diff = np.abs(prob_lr3 - prob_lr4)

print(f"\nMean difference: {diff.mean():.4f}")
print(f"Median difference: {np.median(diff):.4f}")
print(f"95th percentile: {np.percentile(diff, 95):.4f}")

Maximum absolute difference: 0.0510

Mean difference: 0.0111
Median difference: 0.0070
95th percentile: 0.0345


*This helper function was created compare the same classification metrics for different models*

In [37]:
def add_model_results(results_df, model_name, y_true, y_pred, y_prob):
    row = {
        "Model": [model_name],
        "Accuracy": [accuracy_score(y_true, y_pred)],
        "Precision": [precision_score(y_true, y_pred)],
        "Recall": [recall_score(y_true, y_pred)],
        "F1": [f1_score(y_true, y_pred)],
        "F2": [fbeta_score(y_true, y_pred, beta=2)],
        "ROC AUC": [roc_auc_score(y_true, y_prob)]
    }

    results_df.loc[len(results_df)] = row
    return results_df

*For this project classification metrics for the models, Random Forest and Logistic Regression, are added to a Dataframe and downloaded for comparison.*

In [38]:
results_df = pd.DataFrame(
    columns=[
        "Model",
        "Recall",
        "Precision",
        "F1",
        "F2",
        "ROC AUC"
    ]
)

results_df = add_model_results(
    results_df,
    "Logistic Regression",
    y_test,
    y_pred_lr,
    lr_model.predict_proba(X_test)[:, 1]
)

results_df = add_model_results(
    results_df,
    "Random Forest",
    y_test,
    y_pred_rf,
    best_rf.predict_proba(X_test)[:, 1]
)

*The results show that Logistic Regression was just a bit better with all values sans precision. McNemar's test also showed no statistically significant difference in overall classification correctness. In addition, feature engineering appears to have transformed the underlying operational conditions into predictors that Logistic Regression could represent effectively.*

*Therefore, I selected Logistic Regression because it achieved comparable performance with a simpler and more interpretable model.*

In [39]:
print(results_df)

                   Model                Recall             Precision  \
0  [Logistic Regression]  [0.7846153846153846]  [0.3695652173913043]   
1        [Random Forest]  [0.7538461538461538]  [0.3712121212121212]   

                      F1                    F2               ROC AUC  
0   [0.5024630541871922]  [0.6407035175879398]   [0.832089708656466]  
1  [0.49746192893401014]               [0.625]  [0.8247118004611192]  


*Save model classification results to a CSV file*

In [40]:
import os

os.makedirs(MODELS_PATH, exist_ok=True)

results_df.to_csv(
    os.path.join(MODELS_PATH, "model_metrics.csv"),
    index=False
)

*Dump files*


In [ ]:
# Uncomment to install
#!pip install joblib

In [43]:
import joblib, pickle

# Logistic regression model
model_file = os.path.join(MODELS_PATH, "logistic_backlog_pipeline.pkl")
with open(model_file, "wb") as file:
    pickle.dump(lr_model2, file)

print(f"Model saved to: {model_file}")

deployment_metadata = {
    "model_features": [
        "work_pressure",
        "packers_assigned",
        "bottleneck_flag"
    ],
    "positive_class": "Backlog Risk",
    "classification_threshold": 0.50,
    "engineered_features": {"work_pressure": "backlog_units / planned_work"
    }
}

metadata_file = os.path.join(MODELS_PATH, "deployment_metadata.pkl")
with open(metadata_file, "wb") as file:
    pickle.dump(deployment_metadata, file)



Model saved to: /content/drive/MyDrive/warehouse-bottleneck-analyzer/models/logistic_backlog_pipeline.pkl
